In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns


def download_data(tickers: list, start_date: str, end_date: str) -> pd.DataFrame:



    print(f"Downloading data for {len(tickers)} assets from {start_date} to {end_date}...")

    raw_data = yf.download(tickers, start=start_date, end=end_date)

    try:
        data = raw_data['Adj Close']

    except KeyError:
        data = raw_data['Close']

    data = data.ffill().dropna()

    print("Data download complete.")

    return data


def calculate_returns(prices: pd.DataFrame) -> pd.DataFrame:

   

    returns = prices.pct_change().dropna()

    return returns


def compute_portfolio_performance(weights, mean_returns, cov_matrix, trading_days=252):

   

    returns = np.sum(mean_returns * weights) * trading_days

    std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(trading_days)

    return returns, std_dev


def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.0):

    

    p_ret, p_vol = compute_portfolio_performance(weights, mean_returns, cov_matrix)

    return - (p_ret - risk_free_rate) / (p_vol + 1e-9)


def optimize_portfolio(returns: pd.DataFrame, risk_free_rate: float = 0.05, max_weight: float = 0.40):

    

    num_assets = len(returns.columns)

    mean_returns = returns.mean()

    cov_matrix = returns.cov()

    init_guess = num_assets * [1. / num_assets]

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

    bounds = tuple((0.0, max_weight) for _ in range(num_assets))

    opt_result = minimize(
        negative_sharpe_ratio,
        init_guess,
        args=(mean_returns, cov_matrix, risk_free_rate),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    if not opt_result.success:

        return np.array(init_guess)

    return opt_result.x


def run_walk_forward(
    returns: pd.DataFrame,
    train_days: int = 126,
    test_days: int = 21,
    risk_free_rate: float = 0.05,
    max_weight: float = 0.40,
    tc_bps: float = 10.0
):

    

    print("Starting walk-forward backtest...")

    n_days = len(returns)

    dates = returns.index

    dynamic_portfolio_returns = pd.Series(dtype=float)

    static_portfolio_returns = pd.Series(dtype=float)

    weight_history = pd.DataFrame(columns=returns.columns)

    turnover_history = []

    initial_train_window = returns.iloc[0:train_days]

    static_weights = optimize_portfolio(initial_train_window, risk_free_rate, max_weight)

    print("Static Portfolio Weights computed.")

    current_weights = None

    tc_decimal = tc_bps / 10000.0

    for i in range(train_days, n_days, test_days):

        train_window = returns.iloc[i - train_days : i]

        end_idx = min(i + test_days, n_days)

        test_window = returns.iloc[i : end_idx]

        if len(test_window) == 0:
            break

        new_weights = optimize_portfolio(train_window, risk_free_rate, max_weight)

        if current_weights is not None:

            turnover = np.sum(np.abs(new_weights - current_weights))

        else:

            turnover = 1.0

        turnover_history.append({'Date': dates[i], 'Turnover': turnover})

        transaction_cost = turnover * tc_decimal

        period_returns_dyn = test_window.dot(new_weights)

        period_returns_stat = test_window.dot(static_weights)

        period_returns_dyn.iloc[0] -= transaction_cost

        dynamic_portfolio_returns = pd.concat([dynamic_portfolio_returns, period_returns_dyn])

        static_portfolio_returns = pd.concat([static_portfolio_returns, period_returns_stat])

        weight_history.loc[dates[i]] = new_weights

        current_weights = new_weights

    print("Backtest complete.")

    dynamic_portfolio_returns.index = returns.index[train_days:]

    static_portfolio_returns.index = returns.index[train_days:]

    turnover_df = pd.DataFrame(turnover_history).set_index('Date')

    return dynamic_portfolio_returns, static_portfolio_returns, weight_history, turnover_df


def calculate_metrics(returns: pd.Series, risk_free_rate=0.05, trading_days=252):

    """Calculates standard quantitative performance metrics."""

    compounded_growth = (1 + returns).prod()

    n_years = len(returns) / trading_days

    cagr = (compounded_growth ** (1 / n_years)) - 1

    volatility = returns.std() * np.sqrt(trading_days)

    ann_return = returns.mean() * trading_days

    sharpe = (ann_return - risk_free_rate) / volatility

    cumulative = (1 + returns).cumprod()

    rolling_max = cumulative.cummax()

    drawdown = (cumulative - rolling_max) / rolling_max

    max_drawdown = drawdown.min()

    return {
        "CAGR": f"{cagr:.2%}",
        "Volatility": f"{volatility:.2%}",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "Max Drawdown": f"{max_drawdown:.2%}"
    }


def plot_performance(dynamic_rets: pd.Series, static_rets: pd.Series):

    """Plots cumulative returns comparison."""

    plt.figure(figsize=(12, 6))

    plt.plot((1 + dynamic_rets).cumprod(), label='Dynamic Walk-Forward (After Costs)', color='blue')

    plt.plot((1 + static_rets).cumprod(), label='Static Assignment Model', color='gray', linestyle='--')

    plt.title('Cumulative Portfolio Performance: Dynamic vs Static')

    plt.ylabel('Cumulative Return')

    plt.xlabel('Date')

    plt.legend()

    plt.grid(True, alpha=0.3)

    plt.tight_layout()

    plt.show()


def plot_drawdown(dynamic_rets: pd.Series):

    """Plots the drawdown curve for the dynamic portfolio."""

    cumulative = (1 + dynamic_rets).cumprod()

    rolling_max = cumulative.cummax()

    drawdown = (cumulative - rolling_max) / rolling_max

    plt.figure(figsize=(12, 4))

    plt.fill_between(drawdown.index, drawdown, 0, color='red', alpha=0.3)

    plt.plot(drawdown, color='red', linewidth=1)

    plt.title('Dynamic Portfolio Drawdown')

    plt.ylabel('Drawdown')

    plt.xlabel('Date')

    plt.grid(True, alpha=0.3)

    plt.tight_layout()

    plt.show()


def plot_weight_evolution(weight_history: pd.DataFrame):

    """Plots the evolution of asset weights over time (Stacked Area)."""

    plt.figure(figsize=(12, 6))

    weight_history_plot = weight_history.resample('D').ffill().dropna()

    plt.stackplot(weight_history_plot.index, weight_history_plot.T, labels=weight_history_plot.columns, alpha=0.8)

    plt.title('Dynamic Portfolio Weight Evolution Over Time')

    plt.ylabel('Weight')

    plt.xlabel('Date')

    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

    plt.margins(0, 0)

    plt.tight_layout()

    plt.show()


TICKERS = [
    'ICICIBANK.NS',
    'INDUSTOWER.NS',
    'HAL.NS',
    'ITC.NS',
    'TCS.NS',
    'GOLDBEES.NS'
]


START_DATE = '2020-01-01'

END_DATE = '2025-05-30'

TRAIN_DAYS = 126

REBALANCE_DAYS = 21

MAX_WEIGHT_CAP = 0.40

RISK_FREE_RATE = 0.05

TRANSACTION_COST_BPS = 10.0


def main():

    print("--- SAPM DYNAMIC PORTFOLIO SYSTEM ---")

    prices = download_data(TICKERS, START_DATE, END_DATE)

    returns = calculate_returns(prices)

    dyn_returns, stat_returns, weights, turnover = run_walk_forward(
        returns=returns,
        train_days=TRAIN_DAYS,
        test_days=REBALANCE_DAYS,
        risk_free_rate=RISK_FREE_RATE,
        max_weight=MAX_WEIGHT_CAP,
        tc_bps=TRANSACTION_COST_BPS
    )

    print("\n--- PERFORMANCE METRICS (Out-of-Sample) ---")

    dyn_metrics = calculate_metrics(dyn_returns, RISK_FREE_RATE)

    stat_metrics = calculate_metrics(stat_returns, RISK_FREE_RATE)

    metrics_df = pd.DataFrame({
        'Dynamic Walk-Forward': dyn_metrics,
        'Static Assignment Model': stat_metrics
    })

    print(metrics_df.to_markdown())

    avg_turnover = turnover['Turnover'].mean()

    print(f"\nAverage Rebalance Turnover: {avg_turnover:.2%}")

    plot_performance(dyn_returns, stat_returns)

    plot_drawdown(dyn_returns)

    plot_weight_evolution(weights)


if __name__ == "__main__":
    main()

--- SAPM DYNAMIC PORTFOLIO SYSTEM ---


[****************      33%                       ]  2 of 6 completedFailed to get ticker 'GOLDBEES.NS' reason: Failed to perform, curl: (28) Connection timed out after 10011 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  6 of 6 completed


Data download complete.
Starting walk-forward backtest...
Static Portfolio Weights computed.
Backtest complete.

--- PERFORMANCE METRICS (Out-of-Sample) ---


ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.